In [4]:
! pip install -q --upgrade pip jedi
! pip install -q llama-index llama-index-readers-file llama-index-llms-google-genai llama-index-embeddings-google-genai

In [3]:
!pip install nbformat==5.10.4 nbconvert==7.16.6

  Using cached nbconvert-7.16.6-py3-none-any.whl.metadata (8.5 kB)
Using cached nbconvert-7.16.6-py3-none-any.whl (258 kB)
  Attempting uninstall: nbconvert
    Found existing installation: nbconvert 7.17.0
    Uninstalling nbconvert-7.17.0:
      Successfully uninstalled nbconvert-7.17.0


In [2]:
import nbformat
import nbconvert

print("nbformat:", nbformat.__version__)
print("nbconvert:", nbconvert.__version__)

nbformat: 5.10.4
nbconvert: 7.17.0


In [1]:
!pip install -U nbformat nbconvert

In [6]:

%pip install -q  nest-asyncio
import nest_asyncio
nest_asyncio.apply()
import os
from google.colab import userdata
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings

os.environ['gemini_chatbot'] = userdata.get('gemini_chatbot')
api= os.environ['gemini_chatbot']

In [15]:
#print(api)

In [9]:
llm = GoogleGenAI(
    model='gemini-2.5-flash-lite',
    api_key=api
)

Settings.llm =llm

1) Envie arquivo para a base de conhecimento utilizando a técnica RAG

Cria uma pasta chamada documentos no Colab e envie seus arquivos para lá

In [10]:
from google.colab import files
import os
os.makedirs("/contents/documentos",exist_ok=True)
print('Pasta criada em /content/documentos')

Pasta criada em /content/documentos


In [12]:
#Leitura dos arquivos
from llama_index.core import SimpleDirectoryReader
documentos = SimpleDirectoryReader(input_dir='/content/documentos')

In [13]:
docs = documentos.load_data()
print(f'Quantidade de documentos carregados {len(docs)}')

Quantidade de documentos carregados 3


In [14]:
# Exibindo os metadados do documento
print(docs[0].get_metadata_str())

page_label: 1
file_name: doc_info_agricultura.pdf
file_path: /content/documentos/doc_info_agricultura.pdf
file_type: application/pdf
file_size: 30901
creation_date: 2026-04-01
last_modified_date: 2026-04-01


In [16]:
#Quebrando o documento em pedaços (nodes)
#importando a biblioteca
from llama_index.core.node_parser import SentenceSplitter
node_parser=SentenceSplitter(chunk_size=1200) # tamanho da divisao
# Fazer a divisao do documento carregado com base no chunk size
nodes = node_parser.get_nodes_from_documents(docs,show_progress = True)
print(f'Quantidade de nodes gerados: {len(nodes)}')


Parsing nodes:   0%|          | 0/3 [00:00<?, ?it/s]

Quantidade de nodes gerados: 3


In [17]:
# Configurando o LLM Gemini e o modelo de embeddings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings

llm = GoogleGenAI(
    model = 'gemini-2.5-flash-lite',
    api_key = api
)

embed_model = GoogleGenAIEmbedding(
    model='gemini-embedding-001',
    api_key=api
)

Settings.llm = llm
Settings.embed_model = embed_model
print('LLM e embeddings configurados')

LLM e embeddings configurados


2) Criando o indice vetorial, esse indice é criado sem o Chroma DB, diretamente em memoria para simplificar a execução no Colab


In [18]:
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex(nodes,embed_model=embed_model)
print('Indice criado com sucesso')

Indice criado com sucesso


In [29]:
from llama_index.core.base.embeddings.base import similarity
# Consulta simples no documento
query_engine = index.as_query_engine(llm=llm,similarity_top_k=1)

In [28]:
chat_engine = index.as_chat_engine(llm=llm,similarity_top_k=1)


4) Modo Chat com memória resumida


In [26]:
from llama_index.core.memory import ChatSummaryMemoryBuffer
memory = ChatSummaryMemoryBuffer(llm=llm,token_limit=256)
chat_engine= index.as_chat_engine(
    chat_mode = 'context',
    llm = llm,
    memory = memory,
    system_prompt=(
        'Você é especialista em preparo do solo para plantio'
        'Sua função é tirar dúvidas de forma simpática e natural sobre o plantio do solo'
    )
)

In [27]:
memory.get()

[]

In [ ]:
# Reset do chat
chat_engine.reset()
print('Chat resetado')

In [30]:
# Interface para o chatbot

! pip install -q ipywidgets

In [21]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from google.genai.errors import ClientError

In [24]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Se ClientError vier de alguma lib específica, mantenha o import correto.
# Ex:
# from google.api_core.exceptions import ClientError

# Output
chat_output = widgets.Output(
    layout={
        "border": "1px solid #ccc",
        "padding": "10px",
        "height": "400px",
        "overflow_y": "auto"
    }
)

# Input
input_box = widgets.Text(
    placeholder="Digite sua pergunta sobre preparo do solo para plantio",
    description="Pergunta:",
    layout=widgets.Layout(width="80%")
)

# Botões
send_button = widgets.Button(
    description="Enviar",
    button_style="info"
)

clear_button = widgets.Button(
    description="Limpar",
    button_style="warning"
)

# Histórico
history = []

def responder(pergunta):
    try:
        response = chat_engine.chat(pergunta)
        return getattr(response, "response", str(response))

    except ClientError as e:
        erro = str(e)

        if "429" in erro or "RESOURCE_EXHAUSTED" in erro:
            try:
                retriever = index.as_retriever(similarity_top_k=1)
                nodes = retriever.retrieve(pergunta)

                if nodes:
                    partes = []
                    for i, node in enumerate(nodes, 1):
                        texto = getattr(node, "text", str(node))
                        partes.append(f"**Trecho {i}:**\n\n{texto[:500]}")

                    return (
                        "A API Gemini atingiu o limite no momento. "
                        "Encontrei estes trechos locais:\n\n"
                        + "\n\n---\n\n".join(partes)
                    )

                return "A API Gemini atingiu o limite e não encontrei trechos locais relevantes."

            except Exception as erro_retriever:
                return f"A API Gemini atingiu o limite e houve erro ao buscar trechos locais: {erro_retriever}"

        return f"Erro na API: {erro}"

    except Exception as e:
        return f"Erro inesperado: {e}"

# Renderizar o chat
def render_chat():
    with chat_output:
        clear_output(wait=True)
        display(Markdown("## Chatbot plantio"))

        if not history:
            display(Markdown("_Nenhuma mensagem ainda._"))

        for role, content in history:
            if role == "user":
                display(Markdown(f"**Você:** {content}"))
            else:
                display(Markdown(f"**Chatbot:** {content}"))

# Envio
def on_send(_):
    pergunta = input_box.value.strip()
    if not pergunta:
        return

    history.append(("user", pergunta))
    resposta = responder(pergunta)
    history.append(("assistant", resposta))

    input_box.value = ""
    render_chat()

# Limpar
def on_clear(_):
    history.clear()
    try:
        chat_engine.reset()
    except Exception:
        pass
    render_chat()

send_button.on_click(on_send)
clear_button.on_click(on_clear)
input_box.on_submit(lambda _: on_send(None))

display(widgets.VBox([
    chat_output,
    widgets.HBox([input_box, send_button, clear_button])
]))

render_chat()

## 1. Introdução

Este projeto implementa um chatbot interativo no Google Colab, focado no preparo do solo para plantio. Ele utiliza a arquitetura Retrieval Augmented Generation (RAG) para fornecer respostas contextualizadas com base em documentos carregados, empregando modelos do Google Gemini.

## 2. Tecnologias Utilizadas

-   **LlamaIndex:** Framework para RAG, indexação de documentos e criação de motores de consulta/chat.
-   **Google GenAI (Generative AI):** Modelos de linguagem (LLM) como `gemini-2.5-flash-lite` e embeddings (`gemini-embedding-001`) via API.
-   **ipywidgets:** Biblioteca Python para construção de interfaces de usuário interativas no Jupyter/Colab.
-   **nbformat, nbconvert:** Ferramentas para manipulação de notebooks Jupyter.
-   **nest_asyncio:** Usado para permitir a execução de código assíncrono em ambientes Jupyter.

## 3. Estrutura do Projeto

O projeto segue uma estrutura modular:
1.  Instalação de dependências.
2.  Configuração da chave de API para o Gemini.
3.  Criação de diretório para documentos e carregamento de arquivos PDF.
4.  Processamento dos documentos, dividindo-os em 'nodes' (pedaços).
5.  Configuração dos modelos LLM e de embeddings.
6.  Criação de um índice vetorial a partir dos 'nodes'.
7.  Inicialização do motor de chat com memória e um *system prompt*.
8.  Desenvolvimento de uma interface de usuário interativa para o chatbot.

## 4. Implementação do Projeto

### 4.1 Inicialização do cliente da API

O cliente da API é inicializado usando `GoogleGenAI` e `GoogleGenAIEmbedding` do LlamaIndex, configurando o `api_key` obtido das variáveis de ambiente (`os.environ['gemini_chatbot']`).

```python
llm = GoogleGenAI(model='gemini-2.5-flash-lite', api_key=api)
embed_model = GoogleGenAIEmbedding(model='gemini-embedding-001', api_key=api)
Settings.llm = llm
Settings.embed_model = embed_model
```

### 4.2 Função do Chatbot

A funcionalidade do chatbot é encapsulada pelo `chat_engine` do LlamaIndex, configurado com:
-   `chat_mode='context'` para usar o contexto dos documentos.
-   `memory=ChatSummaryMemoryBuffer(llm=llm, token_limit=256)` para manter a memória da conversa.
-   `system_prompt` definindo o persona do chatbot como especialista em preparo do solo.

### 4.3 Retorno da resposta

As respostas são geradas chamando `chat_engine.chat(pergunta)`, que interage com o modelo Gemini e o índice de documentos para fornecer uma resposta relevante à consulta do usuário.

### 4.4 Tratamento de erros

A função `responder` trata `ClientError` (especialmente `429` ou `RESOURCE_EXHAUSTED` para limites de API). Em caso de limite atingido, ela tenta recuperar e exibir trechos relevantes dos documentos locais usando o retriever do índice, oferecendo uma resposta alternativa ao usuário.

## 5. Interface do Usuário

Implementada com `ipywidgets` para uma experiência interativa no Colab:
-   `chat_output`: Exibe o histórico da conversa (`widgets.Output`).
-   `input_box`: Campo para o usuário digitar perguntas (`widgets.Text`).
-   `send_button`: Envia a pergunta (`widgets.Button`).
-   `clear_button`: Limpa o histórico do chat (`widgets.Button`).

Os elementos são organizados usando `widgets.VBox` e `widgets.HBox`.

## 6. Conclusão

O projeto demonstra a criação de um chatbot RAG eficaz para informações especializadas, utilizando a capacidade de linguagem do Gemini e a recuperação de informações do LlamaIndex. A interface amigável e o tratamento de erros aprimoram a experiência do usuário, tornando-o uma ferramenta prática para consultas sobre preparo do solo.